In [1]:
import subprocess
import os
import csv
import pandas as pd
from bayes_opt import BayesianOptimization
from pathlib import Path
import numpy as np
from simulation_functions import call_vot_simulation, intialise_java, call_stt_simulation,STT_INIT_POINTS, STT_N_ITER
from scipy import stats

VOT config

In [2]:
#VOT config
defaultAlpha_bounds  = (-5,5)
default_beta_bounds = (-1, -0.001)
vot_pbounds = {'alphaWalk' :(0,0) ,
           'alphaBike' : defaultAlpha_bounds,
           'alphaCarDriver' : defaultAlpha_bounds,
           'alphaCarPassenger' : defaultAlpha_bounds,
           'alphaBus' : defaultAlpha_bounds,
           'alphaTrain' : defaultAlpha_bounds,
           'betaChangesTransport': default_beta_bounds,
           'betaCostLow':(-0.4,-0.2) ,
           'betaCostMed': (-0.2,-0.2),
           'betaCostHigh': (-0.2,-0.01),
           'weightWalk': (1,2),
           'weightWait': (1,2),
           'weightFeeder': (1,2), 
           'weightVotCosts' : (1,1),
           'weightTangibleCosts' : (1,1)
           } 

STT config

In [3]:
stt_pbounds = {
    'alphaWalk' :(0,0) ,
    'alphaBike' : defaultAlpha_bounds,
    'alphaCarDriver' : defaultAlpha_bounds,
    'alphaCarPassenger' : defaultAlpha_bounds,
    'alphaBus' : defaultAlpha_bounds,
    'alphaTrain' : defaultAlpha_bounds,
    'betaTimeWalk' : (-0.04,-0.04),
    'betaTimeBike' : (-0.03,-0.03),
    'betaTimeCarDriver': (-0.02,-0.02),
    'betaTimeCarPassenger' : (-0.02,-0.02),
    'betaTimeBus' : (-0.02,-0.02),
    'betaTimeTrain' : (-0.02,-0.02),
    'betaCostCarDriver': (-0.15,-0.15),
    'betaCostCarPassenger' : (-0.15,-0.15),
    'betaCostBus': (-0.1,-0.1),
    'betaCostTrain': (-0.1,-0.1),
    'betaTimeWalkTransport': (-0.03,-0.03),
    'betaChangesTransport': (-0.3,-0.3)}

In [4]:
intialise_java()

Java Version Output:
 openjdk version "11.0.20.1" 2023-08-24 LTS
OpenJDK Runtime Environment Microsoft-8297088 (build 11.0.20.1+1-LTS)
OpenJDK 64-Bit Server VM Microsoft-8297088 (build 11.0.20.1+1-LTS, mixed mode)



In [5]:

rq_1_config = {
    "config_file": "src/main/resources/config_DHZW_rq1-1.toml",
    "parameterset": "src/main/resources/baseline_parameterset/parameterset_rq1-1.csv",
    "parameterset_write_folder" : "../7-simulation-Sim-2APL/",
    "output_path": 'src/main/resources/baseline_parameterset/output-rq1-1.csv',
    "other_args" : "--use_random_seed true"
}
rq1_1_optimiser = BayesianOptimization(
    f=lambda **params: call_stt_simulation(config=rq_1_config, **params),
    pbounds=stt_pbounds,
    random_state=1,
)

rq1_1_optimiser.maximize(
    init_points=STT_INIT_POINTS,
    n_iter=STT_N_ITER
)

|   iter    |  target   | alphaWalk | alphaBike | alphaC... | alphaC... | alphaBus  | alphaT... | betaTi... | betaTi... | betaTi... | betaTi... | betaTi... | betaTi... | betaCo... | betaCo... | betaCo... | betaCo... | betaTi... | betaCh... |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| 1         | -31.44133 | 0.0       | 2.2032449 | -4.998856 | -1.976674 | -3.532441 | -4.076614 | -0.04     | -0.03     | -0.02     | -0.02     | -0.02     | -0.02     | -0.15     | -0.15     | -0.1      | -0.1      | -0.03     | -0.3      |
| 2         | -17.18816 | 0.0       | -3.018985 | 3.0074456 | 4.6826157 | -1.865758 | 1.9232261 | -0.04     | -0.03     | -0.02     | -0.02     | -0.02     | -0.02     | -0.15     | -0.15     | -0.1      | -0.1      | -0.03     | -0.3      |
| 3         | -22.26021 | 0.0   

In [6]:
print(rq1_1_optimiser.max)

{'target': np.float64(-8.06016040698408), 'params': {'alphaWalk': np.float64(0.0), 'alphaBike': np.float64(-0.8389397099319995), 'alphaCarDriver': np.float64(5.0), 'alphaCarPassenger': np.float64(2.7799717505275465), 'alphaBus': np.float64(0.805098644163749), 'alphaTrain': np.float64(2.1944519303812626), 'betaTimeWalk': np.float64(-0.04), 'betaTimeBike': np.float64(-0.03), 'betaTimeCarDriver': np.float64(-0.02), 'betaTimeCarPassenger': np.float64(-0.02), 'betaTimeBus': np.float64(-0.02), 'betaTimeTrain': np.float64(-0.02), 'betaCostCarDriver': np.float64(-0.15), 'betaCostCarPassenger': np.float64(-0.15), 'betaCostBus': np.float64(-0.1), 'betaCostTrain': np.float64(-0.1), 'betaTimeWalkTransport': np.float64(-0.03), 'betaChangesTransport': np.float64(-0.3)}}


In [7]:
rq_1_config_no_seed = rq_1_config
rq_1_config_no_seed["other_args"] = "--use_random_seed false"
base_pop_results = []
for i in range(0,30):
    base_pop_results.append(- call_stt_simulation(config=rq_1_config_no_seed,**rq1_1_optimiser.max['params']))


In [8]:
base_pop_results

[8.078856646164647,
 8.069551887294185,
 8.075341126236319,
 8.077668179308883,
 8.072089261165532,
 8.08434725328873,
 8.070135626168561,
 8.07003999884651,
 8.074229639153051,
 8.073871153762488,
 8.076644709455191,
 8.067903638080667,
 8.074645258436167,
 8.07695697572408,
 8.069666279765226,
 8.073238056549226,
 8.075901298340282,
 8.074258667977556,
 8.066514722376704,
 8.072437841187186,
 8.07148870370448,
 8.069260779586159,
 8.067953580518305,
 8.074462078074248,
 8.072890298665042,
 8.076124680057342,
 8.07104170586535,
 8.066528599568002,
 8.070791356185456,
 8.081042173765757]

In [9]:
np.std(base_pop_results)

np.float64(0.00412802775847414)

RQ1.2

In [10]:

rq_2_config = {
    "config_file": "src/main/resources/config_DHZW_rq1-2.toml",
    "parameterset": "src/main/resources/baseline_parameterset/parameterset_rq1-2.csv",
    "parameterset_write_folder" : "../7-simulation-Sim-2APL/",
    "output_path": 'src/main/resources/baseline_parameterset/output-rq1-2.csv',
    "other_args" : "--use_random_seed true"
}
rq1_2_optimiser = BayesianOptimization(
    f=lambda **params: call_stt_simulation(config=rq_2_config, **params),
    pbounds=stt_pbounds,
    random_state=1,
)

rq1_2_optimiser.maximize(
    init_points=STT_INIT_POINTS,
    n_iter=STT_N_ITER,
)

|   iter    |  target   | alphaWalk | alphaBike | alphaC... | alphaC... | alphaBus  | alphaT... | betaTi... | betaTi... | betaTi... | betaTi... | betaTi... | betaTi... | betaCo... | betaCo... | betaCo... | betaCo... | betaTi... | betaCh... |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| 1         | -31.44413 | 0.0       | 2.2032449 | -4.998856 | -1.976674 | -3.532441 | -4.076614 | -0.04     | -0.03     | -0.02     | -0.02     | -0.02     | -0.02     | -0.15     | -0.15     | -0.1      | -0.1      | -0.03     | -0.3      |
| 2         | -15.40485 | 0.0       | -3.018985 | 3.0074456 | 4.6826157 | -1.865758 | 1.9232261 | -0.04     | -0.03     | -0.02     | -0.02     | -0.02     | -0.02     | -0.15     | -0.15     | -0.1      | -0.1      | -0.03     | -0.3      |
| 3         | -21.14509 | 0.0   

In [11]:
rq_2_config_no_seed = rq_2_config
rq_2_config_no_seed["other_args"] = "--use_random_seed false"
gen_synth_pop_results = []
for i in range(0,30):
    gen_synth_pop_results.append(- call_stt_simulation(config=rq_2_config_no_seed,**rq1_2_optimiser.max['params']))

Significance

In [12]:
baseline = np.array(base_pop_results)
improved = np.array(gen_synth_pop_results)

t_stat, p_value = stats.ttest_ind(baseline, improved)

print(f"P-value: {p_value}")
if p_value < 0.05:
    print("Statistically significant difference.")

P-value: 2.778937591953034e-166
Statistically significant difference.


In [13]:
#Confidence intervals
n1, n2 = len(baseline), len(improved)
mean1, mean2 = np.mean(baseline), np.mean(improved)
diff = mean1 - mean2


se_diff = np.sqrt(np.var(baseline, ddof=1)/n1 + np.var(improved, ddof=1)/n2)

df = n1 + n2 - 2
t_crit = stats.t.ppf(0.975, df)

lower_bound = diff - (t_crit * se_diff)
upper_bound = diff + (t_crit * se_diff)

print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")
print(f"95% CI of the difference: [{lower_bound:.4f}, {upper_bound:.4f}]")

t-statistic: 5238.4705
p-value: 0.0000
95% CI of the difference: [6.5379, 6.5429]
